# Student Dropout Data Combining and Preparation

This notebook combines multiple student data files into a single dataset for analysis. It handles data loading, cleaning, and merging operations to create a comprehensive dataset for student dropout prediction.

## Table of Contents
1. [Environment Setup](#environment-setup)
2. [Data Loading](#data-loading)
3. [Data Cleaning](#data-cleaning)
4. [Data Merging and Analysis](#data-merging-and-analysis)
5. [Final Dataset Export](#final-dataset-export)

---

## 1. Environment Setup {#environment-setup}

Setting up the required libraries and configurations.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display
from itertools import combinations
from collections import Counter, defaultdict

### Display Settings Configuration

Configure pandas display options for better data visualization during analysis.

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

---

## 2. Data Loading {#data-loading}

Load CSV files from the raw data directory with automatic encoding detection.

In [ ]:
# Set up data directories
current_user = os.getlogin()
DATA_DIR = Path(rf"/home/{current_user}/local-share")
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "processed"
PROC_DIR = Path(rf"/home/{current_user}/local-share")

# Helper function: read CSV with encoding fallback
def read_csv_smart(path: Path, **kwargs) -> pd.DataFrame:
    """Read CSV with UTF-8-BOM first, fallback to Latin-1."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kwargs)

# Load all CSV files from RAW_DIR
def load_csvs_flat(base: Path, **kwargs) -> dict[str, pd.DataFrame]:
    """Load CSVs from directory (flat search only)."""
    tables, errors = {}, []
    for p in sorted(base.glob("*.csv")):
        key = p.stem
        try:
            tables[key] = read_csv_smart(p, **kwargs)
        except Exception as e:
            errors.append((str(p), str(e)))
    print(f"Loaded {len(tables)} tables; errors: {len(errors)}")
    if errors:
        for name, err in errors[:10]:
            print(f" - {name} -> {err}")
    return tables

# Load CSV files
tables = load_csvs_flat(RAW_DIR, sep=None, engine="python")

# Print summary
print(f"\nNumber of csv files found: {len(tables)}")
print("List of csv files:")
for i, name in enumerate(tables.keys(), start=1):
    print(f"{i}. {name}")

In [ ]:
# Summary of loaded datasets
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_seq_items", None)

summary = (
    pd.DataFrame([{
        "key": k,
        "rows": len(df),
        "cols": df.shape[1],
        "columns": list(map(str, df.columns))
    } for k, df in tables.items()])
    .sort_values("key")
)

summary

In [ ]:
# Create DataFrames with clean, standardized names
custom_names = {
    "sdo1_distance": "sdo1-helperdata-Euclidean_distance_postcode_to_HU",
    "sdo1_characteristics": "sdo1-student_characteristics",
    "sdo1_previous": "sdo1-student_previous_education",
    "sdo2_skc": "sdo2-skc",
    "sdo2_orientation": "sdo2-student_orientation",
    "sdo4_dsa": "sdo4-DSA_degree_collegeyear_2018-2023",
    "sdo5_degree": "sdo5-student_degree_drop-out_postalcode",
    "sdo6_results": "sdo6-course_results",
    "sdo7_questionaire": "sdo78-student_wellbeing",
    "sdo4_nf": "sdo4-NF_degree_collegeyear"
}

for clean_name, source_key in custom_names.items():
    df = tables.get(source_key)
    if df is None:
        print(f"[warning] '{source_key}' not found in tables.")
        continue
    globals()[clean_name] = df
    print(f"Created DataFrame: {clean_name} from '{source_key}' with {len(df)} rows")

---

## 3. Data Cleaning {#data-cleaning}

Remove empty rows and unnamed columns from all datasets to ensure data quality.

### 3.1 Removing Empty Rows

Drop rows that are entirely empty (all NaN values) across all columns.

In [ ]:
df_names = [
    "sdo1_distance", "sdo1_characteristics", "sdo1_previous",
    "sdo2_skc", "sdo2_orientation", "sdo4_dsa",
    "sdo5_degree", "sdo6_results", "sdo7_questionaire", "sdo4_nf"
]

def _drop_fully_empty_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """Convert whitespace to NaN in text columns and drop fully empty rows."""
    obj_cols = df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols):
        df = df.copy()
        df[obj_cols] = df[obj_cols].replace(r"^\s*$", np.nan, regex=True)
    
    empty_mask = df.isna().all(axis=1)
    n_empty = int(empty_mask.sum())
    return df.loc[~empty_mask].copy() if n_empty else df, n_empty

# Drop empty rows from all datasets
for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found; skipping.")
        continue
    
    before = len(df)
    cleaned, dropped = _drop_fully_empty_rows(df)
    globals()[name] = cleaned
    print(f"{name}: dropped {dropped} rows ({before} → {len(cleaned)})")

# Verify no empty rows remain
for name in df_names:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        assert not df.isna().all(axis=1).any(), f"{name} still has empty rows."

### 3.2 Removing Unnamed Columns

Drop unnamed columns (e.g., `Unnamed: 0`) that are typically created from exported indices or blank headers.

In [ ]:
df_names = [
    "sdo1_distance", "sdo1_characteristics", "sdo1_previous",
    "sdo2_skc", "sdo2_orientation", "sdo4_dsa",
    "sdo5_degree", "sdo6_results", "sdo7_questionaire", "sdo4_nf"
]

def _unnamed_cols(df: pd.DataFrame):
    """Identify unnamed columns (blank or 'Unnamed: N')."""
    cols = []
    for c in df.columns:
        s = str(c)
        if s.strip() == "" or s.lower().startswith("unnamed"):
            cols.append(c)
    return cols

# Drop unnamed columns from all datasets
for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        print(f"[warning] '{name}' not found; skipping.")
        continue
    
    to_drop = _unnamed_cols(df)
    if to_drop:
        globals()[name] = df.drop(columns=to_drop)
        print(f"{name}: dropped {len(to_drop)} unnamed columns!")
    else:
        print(f"{name}: no unnamed columns found.")

---

## 4. Data Analysis and Merging {#data-merging-and-analysis}

This section analyzes data completeness, validates schema, prepares datasets for merging, and performs the actual data integration using both single-key and composite-key strategies.

### 4.1 Schema Validation and Data Overview

Verify that each loaded dataset matches the expected structure and identify primary keys.

#### 4.1.1 Create Multiple Previous Education Indicator

In [ ]:
# Create Multiple_Previous_Education indicator
if 'SINH_ID' in sdo1_previous.index.names:
    sdo1_previous = sdo1_previous.reset_index(drop=True)

prev_counts = (
    sdo1_previous
    .groupby('SINH_ID')
    .size()
    .reset_index(name='prev_record_count')
)

prev_counts['Multiple_Previous_Education'] = (prev_counts['prev_record_count'] > 1).astype(int)

sdo1_previous = sdo1_previous.merge(
    prev_counts[['SINH_ID', 'Multiple_Previous_Education']],
    on='SINH_ID',
    how='left'
)

print("Multiple_Previous_Education counts:")
print(sdo1_previous['Multiple_Previous_Education'].value_counts())

#### 4.1.2 Primary Key Validation

Verify data schema and identify primary keys for all datasets.

In [ ]:
# Primary key validation and normalization
clean_names = list(custom_names.keys())
df_registry = {n: globals()[n] for n in clean_names if n in globals() and isinstance(globals()[n], pd.DataFrame)}

def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """Detect column matching SINH_ID (case/spacing insensitive)."""
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def search_composite_key(df: pd.DataFrame, max_combos: int = 5000):
    """Find valid composite key (pairs only)."""
    tested = 0
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        null_rows = int(df[list(cols)].isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows
    return None, None, None

# Normalize SINH_ID headers and test PKs
rename_log, pk_rows, dup_samples = [], [], {}

for ds_name, df in df_registry.items():
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))
    
    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)
        
        pk_rows.append({
            "dataset": ds_name, "has_SINH_ID": True, "pk_type": "single",
            "pk_columns": ("SINH_ID",), "null_rows": nulls, "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid)
        })
        
        if not is_valid and dup_rows > 0:
            dup_samples[ds_name] = (
                df["SINH_ID"].value_counts(dropna=False)
                .loc[lambda s: s > 1].head(10)
                .rename("count").reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
    else:
        cand_cols, nulls, dup_rows = search_composite_key(df, max_combos=5000)
        pk_rows.append({
            "dataset": ds_name, "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else None),
            "pk_columns": cand_cols, "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(cand_cols is not None)
        })

pk_report = pd.DataFrame(pk_rows).sort_values(["has_SINH_ID", "dataset"], ascending=[False, True]).reset_index(drop=True)

if rename_log:
    print("Renamed headers to normalize SINH_ID:")
    for ds, old, new in rename_log:
        print(f"  - {ds}: '{old}' -> '{new}'")

print("\nPrimary key validation summary:")
display(pk_report)

for ds_name, dup_df in dup_samples.items():
    print(f"\nDuplicated SINH_IDs in {ds_name}:")
    display(dup_df)

### 4.2 Data Preparation and Cleaning

Handle datasets with invalid primary keys and normalize column names for consistency.

#### 4.2.1 Fix Invalid Primary Keys

Handle datasets where SINH_ID has duplicates or missing values.

##### Fix `sdo1_previous` table:

In [ ]:
# Make SINH_ID unique by keeping latest Final Exam Date per student
df = sdo1_previous.copy()

df["Final Exam Date"] = pd.to_datetime(df["Final Exam Date"], dayfirst=True, errors="coerce")
df["SINH_ID"] = pd.to_numeric(df["SINH_ID"], errors="coerce").round().astype("Int64")
df = df.dropna(subset=["SINH_ID"]).copy()
df = df.sort_values(["SINH_ID", "Final Exam Date"], ascending=[True, True], na_position="first")

sdo1_previous = df.drop_duplicates(subset=["SINH_ID"], keep="last").copy()

assert sdo1_previous["SINH_ID"].isna().sum() == 0, "SINH_ID has NaNs"
assert sdo1_previous.duplicated(subset=["SINH_ID"]).sum() == 0, "SINH_ID not unique"

print(f"sdo1_previous: {len(sdo1_previous)} rows, unique on SINH_ID")

##### Fix `sdo2_orientation` table:

In [ ]:
# Clean sdo2_orientation: convert SINH_ID to Int64 and drop NA rows
sdo2_orientation["SINH_ID"] = pd.to_numeric(sdo2_orientation["SINH_ID"], errors="coerce").round().astype("Int64")
sdo2_orientation = sdo2_orientation.dropna(subset=["SINH_ID"]).copy()
print(f"sdo2_orientation: {len(sdo2_orientation)} rows after dropping NA SINH_ID")

#### 4.2.2 Normalize Column Names

Standardize column names and add dataset prefixes for provenance tracking.

In [ ]:
# Normalize column names: add dataset prefix (idempotent)
df_names = [
    "sdo1_distance", "sdo1_characteristics", "sdo1_previous",
    "sdo2_skc", "sdo2_orientation", "sdo4_dsa",
    "sdo5_degree", "sdo6_results", "sdo7_questionaire", "sdo4_nf"
]

preserve_keys = {"SINH_ID"}

def underscore_words(name: str) -> str:
    """Convert column name to underscored format."""
    s = str(name)
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    s = re.sub(r"([A-Za-z])([0-9])", r"\1_\2", s)
    s = re.sub(r"([0-9])([A-Za-z])", r"\1_\2", s)
    s = re.sub(r"[^0-9A-Za-z]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

preserve_keys_norm = {underscore_words(k) for k in preserve_keys}

for name in df_names:
    df = globals().get(name)
    if not isinstance(df, pd.DataFrame):
        continue
    
    prefix = f"{name}_"
    suffix = f"_{name}"
    current_cols = list(df.columns)
    
    non_key_cols = [c for c in current_cols if underscore_words(c) not in preserve_keys_norm]
    if non_key_cols and all(str(c).startswith(prefix) for c in non_key_cols):
        continue
    
    col_map = {}
    for orig in current_cols:
        base = underscore_words(orig)
        if base.endswith(suffix):
            base = base[:-len(suffix)]
        
        if base.startswith(prefix):
            target = base
        elif base in preserve_keys_norm:
            target = base
            if str(orig).startswith(prefix) or str(orig).endswith(suffix):
                continue
        else:
            target = f"{prefix}{base}"
        
        if orig != target:
            col_map[orig] = target
    
    if not col_map:
        continue
    
    # Handle duplicate targets
    counts = Counter(col_map.values())
    if any(cnt > 1 for cnt in counts.values()):
        seen = defaultdict(int)
        for k, v in list(col_map.items()):
            seen[v] += 1
            if counts[v] > 1 and seen[v] > 1:
                col_map[k] = f"{v}__{seen[v]}"
    
    globals()[name] = df.rename(columns=col_map)
    print(f"{name}: renamed {len(col_map)} columns")

In [ ]:
# Dataset dimensions summary
df_names = [
    "sdo1_distance", "sdo1_characteristics", "sdo1_previous",
    "sdo2_skc", "sdo2_orientation", "sdo4_dsa",
    "sdo5_degree", "sdo6_results", "sdo7_questionaire", "sdo4_nf"
]

summary = [{"dataset": name, "rows": len(df), "cols": df.shape[1]} 
           for name in df_names if isinstance((df := globals().get(name)), pd.DataFrame)]

pd.DataFrame(summary).sort_values("dataset").reset_index(drop=True)

#### 4.2.3 Re-validate Schema After Cleaning

Verify primary keys after handling duplicates and missing values.

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Normalize SINH_ID naming across all datasets and verify primary keys
# per-DataFrame (no mixing). For frames with SINH_ID (in any case/spacing form),
# rename to 'SINH_ID' and test PK (nulls + duplicates). For frames that lack
# SINH_ID (e.g., sdo1_distance, sdo4_dsa), search for a composite key (pairs,
# optionally triples) and report findings with diagnostics.
# -----------------------------------------------------------------------------

# --- 1) Rebuild registry safely (no mixing) ---
clean_names = list(custom_names.keys())
df_registry = {}
for name in clean_names:
    df = globals().get(name, None)
    if isinstance(df, pd.DataFrame):
        df_registry[name] = df
    else:
        print(f"[warning] DataFrame '{name}' not found. Skipping.")

# --- 2) Helpers ---
def _canonicalize(s: str) -> str:
    """Lowercase and remove non-alphanumerics for fuzzy header matching."""
    return re.sub(r'[^a-z0-9]', '', s.lower()) if isinstance(s, str) else s

def find_sinh_id_column(cols) -> str | None:
    """
    Detect a column that semantically matches SINH_ID even if the header's case,
    spacing or punctuation varies, e.g., 'sinh_id', 'Sinh Id', 'SINH-ID'.
    """
    want = "sinhid"
    for c in cols:
        if _canonicalize(c) == want:
            return c
    return None

def robust_min_max(s: pd.Series):
    if is_numeric_dtype(s):
        c = pd.to_numeric(s, errors="coerce")
        return c.min(), c.max()
    if is_datetime64_any_dtype(s):
        c = pd.to_datetime(s, errors="coerce")
        return c.min(), c.max()
    return None, None


def search_composite_key(df: pd.DataFrame, try_triples: bool = False, max_combos: int = 5000):
    """
    Attempt to find a valid composite key (pairs, optionally triples).
    Returns (candidate_cols: tuple|None, null_rows, dup_rows).
    """
    tested = 0

    # Pairs first
    for cols in combinations(df.columns, 2):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    if not try_triples:
        return None, None, None

    # Triples (can be heavy)
    for cols in combinations(df.columns, 3):
        if tested >= max_combos:
            break
        tested += 1
        sub = df[list(cols)]
        null_rows = int(sub.isna().any(axis=1).sum())
        dup_rows = int(df.duplicated(subset=list(cols)).sum())
        if null_rows == 0 and dup_rows == 0:
            return cols, null_rows, dup_rows

    return None, None, None

# --- 3) Normalize SINH_ID headers in-place, then test PKs ---
rename_log = []
pk_rows = []
dup_samples = {}  # dataset -> small df of duplicated key values (if any)

# Add a 'rows' column (rowcount per DataFrame) and show it right after 'dataset'

# ... keep everything above unchanged ...

for ds_name, df in df_registry.items():
    # 3a) Normalize header to 'SINH_ID' if a fuzzy match exists
    match = find_sinh_id_column(df.columns)
    if match and match != "SINH_ID":
        df.rename(columns={match: "SINH_ID"}, inplace=True)
        rename_log.append((ds_name, match, "SINH_ID"))

    # 3b) Test PKs
    n_rows = int(len(df))  # <-- rowcount

    if "SINH_ID" in df.columns:
        nulls = int(df["SINH_ID"].isna().sum())
        dup_rows = int(df.duplicated(subset=["SINH_ID"]).sum())
        is_valid = (nulls == 0) and (dup_rows == 0)

        pk_rows.append({
            "dataset": ds_name,
            "rows": n_rows,                 # <-- add here
            "has_SINH_ID": True,
            "pk_type": "single",
            "pk_columns": ("SINH_ID",),
            "null_rows": nulls,
            "dup_rows": dup_rows,
            "is_valid_pk": bool(is_valid),
        })

        if not is_valid and dup_rows > 0:
            dupe_vals = (
                df["SINH_ID"]
                .value_counts(dropna=False)
                .loc[lambda s: s > 1]
                .head(10)
                .rename("count")
                .reset_index()
                .rename(columns={"index": "SINH_ID"})
            )
            dup_samples[ds_name] = dupe_vals

    else:
        cand_cols, nulls, dup_rows = search_composite_key(df, try_triples=False, max_combos=5000)
        is_valid = cand_cols is not None

        pk_rows.append({
            "dataset": ds_name,
            "rows": n_rows,                 # <-- add here
            "has_SINH_ID": False,
            "pk_type": ("pair" if cand_cols and len(cand_cols) == 2 else
                        "triple" if cand_cols and len(cand_cols) == 3 else None),
            "pk_columns": cand_cols,
            "null_rows": int(nulls) if nulls is not None else None,
            "dup_rows": int(dup_rows) if dup_rows is not None else None,
            "is_valid_pk": bool(is_valid),
        })

# --- 4) Assemble reports (show 'rows' right after 'dataset') ---
pk_report = (
    pd.DataFrame(pk_rows)
      .sort_values(["has_SINH_ID", "dataset"], ascending=[False, True])
      .reset_index(drop=True)
)

# Reorder columns for display: dataset, rows, then the rest
cols_order = ["dataset", "rows", "has_SINH_ID", "pk_type", "pk_columns", "null_rows", "dup_rows", "is_valid_pk"]
pk_report = pk_report.loc[:, cols_order]

print("\nPrimary key validation summary (single SINH_ID where present; composite search otherwise):")
display(pk_report)


##### Decision: Exclude `sdo7_questionaire`

Exclude `sdo7_questionaire` dataframe - it contains too many quality issues and missing data.

In [ ]:
# Remove sdo7_questionaire DataFrame from the global namespace
globals().pop("sdo7_questionaire", None)  # safe even if it doesn't exist

# Optional: free memory aggressively
import gc; gc.collect()

### 4.3 Data Merging Strategy

Combine datasets using a two-stage approach:

1. **Primary Key Merging:** Use SINH_ID for all tables with this primary key
2. **Composite Key Merging:** Use combinations of program codes, years, and postcodes for tables without SINH_ID

#### 4.3.1 First Merge - Primary Key (SINH_ID)

Merge datasets using SINH_ID as the primary key. This creates the `first_merge` dataframe containing:
- sdo5_degree (base table - all students)
- sdo1_characteristics
- sdo1_previous
- sdo2_skc
- sdo2_orientation
- sdo6_results

In [ ]:
# ---------------------------------------------------------------------------
# Build first_merge strictly from:
#   sdo5_degree  (BASE, keep all columns)
#   + sdo1_characteristics
#   + sdo1_previous
#   + sdo2_skc
#   + sdo2_orientation
#   + sdo6_results
# All joins are LEFT on SINH_ID with NO suffixes allowed.
# Adds presence flags at the end.
# ---------------------------------------------------------------------------


def ensure_plain_column_key(df, key="SINH_ID"):
    df = df.copy()
    if isinstance(df.index, pd.MultiIndex):
        if key in (df.index.names or []):
            df = df.reset_index(drop=(key in df.columns))
    elif getattr(df.index, "name", None) == key:
        df = df.reset_index(drop=(key in df.columns))
    return df.rename_axis(None)

def as_int64(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")

def assert_no_overlap(left: pd.DataFrame, right: pd.DataFrame, key: str = "SINH_ID"):
    overlap = (set(left.columns) & set(right.columns)) - {key}
    if overlap:
        raise ValueError(
            f"Refusing to merge because these columns would collide without suffixes: {sorted(overlap)}.\n"
            f"Rename/drop before merging."
        )

def left_merge_no_suffix(left: pd.DataFrame, right: pd.DataFrame, on: str = "SINH_ID"):
    assert_no_overlap(left, right, on)
    before = len(left)
    out = left.merge(right, on=on, how="left", validate="one_to_one")
    assert len(out) == before, f"Row count changed on merge with key '{on}'"
    return out

# ----------------- Base -----------------
first_merge = ensure_plain_column_key(sdo5_degree, "SINH_ID").copy()
first_merge["SINH_ID"] = as_int64(first_merge["SINH_ID"])
base_n = len(first_merge)

# ----------------- Prepare right-hand tables (SINH_ID only) -----------------
char     = ensure_plain_column_key(sdo1_characteristics, "SINH_ID").copy()
prev     = ensure_plain_column_key(sdo1_previous,       "SINH_ID").copy()
skc      = ensure_plain_column_key(sdo2_skc,            "SINH_ID").copy()
orient   = ensure_plain_column_key(sdo2_orientation,    "SINH_ID").copy()
results  = ensure_plain_column_key(sdo6_results,        "SINH_ID").copy()

for df_ in (char, prev, skc, orient, results):
    df_["SINH_ID"] = as_int64(df_["SINH_ID"])

# ----------------- Presence flags (from source availability) -----------------
has_characteristics = set(char["SINH_ID"].dropna())
has_previous        = set(prev["SINH_ID"].dropna())
has_skc             = set(skc["SINH_ID"].dropna())
has_orientation     = set(orient["SINH_ID"].dropna())
has_results         = set(results["SINH_ID"].dropna())

# ----------------- LEFT merges (no suffixes) -----------------
first_merge = left_merge_no_suffix(first_merge, char,    on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, prev,    on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, skc,     on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, orient,  on="SINH_ID")
first_merge = left_merge_no_suffix(first_merge, results, on="SINH_ID")

# ----------------- Presence flags (added at the end) -----------------
first_merge["has_characteristics"] = first_merge["SINH_ID"].isin(has_characteristics).astype("Int8")
first_merge["has_previous"]        = first_merge["SINH_ID"].isin(has_previous).astype("Int8")
first_merge["has_skc"]             = first_merge["SINH_ID"].isin(has_skc).astype("Int8")
first_merge["has_orientation"]     = first_merge["SINH_ID"].isin(has_orientation).astype("Int8")
first_merge["has_results"]         = first_merge["SINH_ID"].isin(has_results).astype("Int8")

print("Base students:", base_n)
print("After merges :", len(first_merge))  # should match base (one_to_one enforced)


#### 4.3.2 Second Merge - Composite Keys

Merge additional datasets using composite keys (program code + year, postcode) to create the final `data` dataframe.

**Datasets to merge:**
- sdo4_dsa (DSA participation)
- sdo4_nf (NF data)
- sdo1_distance (geographical distance data - merged twice for home and previous school locations)

**Step 1: Identify Join Keys**

Define join keys based on business knowledge.

**Step 2: Validate and Normalize Join Keys**

Verify that join key columns have compatible data types and formats before merging.

**Step 3: Pre-Merge Data Quality Check**

Assess missing values in each dataset before merging to understand baseline data quality.

**Step 4: Execute Composite Key Merges**

Perform the actual merging operations with normalized keys and validation.

In [ ]:
# Composite-key merges: DSA/NF indicators and distance joins
# Merge: DSA (program+year), NF (program+year), Distances (PC4 from prev school & home)

import pandas as pd

def norm_year(s: pd.Series) -> pd.Series:
    """Extract 4-digit year from '2023/2024' format."""
    return s.astype("string").str.extract(r"(\d{4})", expand=False)

def norm_prog_code(s: pd.Series) -> pd.Series:
    """Extract digits from program code."""
    return s.astype("string").str.strip().str.extract(r"(\d+)", expand=False)

def norm_postcode_pc4_int(s: pd.Series) -> pd.Series:
    """Convert postcode to PC4 (first 4 digits) as Int64."""
    digits = s.astype("string").str.replace(r"\D+", "", regex=True)
    pc4 = digits.str[:4].where(digits.str.len() >= 4)
    return pc4.astype("Int64")

def assert_no_overlap(left: pd.DataFrame, right: pd.DataFrame, ignore) -> None:
    """Ensure no column collisions before merge."""
    overlap = (set(left.columns) & set(right.columns)) - set(ignore)
    if overlap:
        raise ValueError(f"Column collision: {sorted(overlap)}")

def validate_columns(df_name: str, df: pd.DataFrame, required: list) -> None:
    """Check if required columns exist in dataframe."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{df_name} missing: {missing}")

# Working copies
co, dsa, nf, dist = first_merge.copy(), sdo4_dsa.copy(), sdo4_nf.copy(), sdo1_distance.copy()

# Column definitions
col_spec = {
    'co': {'year': "sdo5_degree_COLLEGEJAAR", 'prog': "sdo5_degree_degree_code",
           'prev_pc': "sdo1_previous_Previous_School_Postal_Code", 'home_pc': "sdo5_degree_POSTCODE"},
    'dsa': {'year': "sdo4_dsa_Collegejaar", 'prog': "sdo4_dsa_CROHO_code"},
    'nf': {'year': "sdo4_nf_Collegejaar", 'prog': "sdo4_nf_CROHO"},
    'dist': {'pc': "sdo1_distance_postcode"}
}

# Validate required columns
for df_name, df_obj, spec_key in [('first_merge', co, 'co'), ('sdo4_dsa', dsa, 'dsa'), 
                                    ('sdo4_nf', nf, 'nf'), ('sdo1_distance', dist, 'dist')]:
    validate_columns(df_name, df_obj, list(col_spec[spec_key].values()))

# Normalize join keys
co["__YEAR__"], co["__PROG__"] = norm_year(co[col_spec['co']['year']]), norm_prog_code(co[col_spec['co']['prog']])
co["__PC4_PREV__"], co["__PC4_HOME__"] = norm_postcode_pc4_int(co[col_spec['co']['prev_pc']]), norm_postcode_pc4_int(co[col_spec['co']['home_pc']])

dsa["__YEAR__"], dsa["__PROG__"] = norm_year(dsa[col_spec['dsa']['year']]), norm_prog_code(dsa[col_spec['dsa']['prog']])
nf["__YEAR__"], nf["__PROG__"] = norm_year(nf[col_spec['nf']['year']]), norm_prog_code(nf[col_spec['nf']['prog']])
dist["__PC4__"] = norm_postcode_pc4_int(dist[col_spec['dist']['pc']])

# Zero-pad program codes
width = int(pd.concat([co["__PROG__"].dropna().map(len), dsa["__PROG__"].dropna().map(len), 
                       nf["__PROG__"].dropna().map(len)], ignore_index=True).max() or 0)
if width > 0:
    for df in [co, dsa, nf]:
        df["__PROG__"] = df["__PROG__"].str.zfill(width)

# Create dimension tables
dsa_keys = (dsa.dropna(subset=["__PROG__", "__YEAR__"]).drop_duplicates(subset=["__PROG__", "__YEAR__"])
            [["__PROG__", "__YEAR__"]].assign(has_dsa=1))
nf_keys = (nf.dropna(subset=["__PROG__", "__YEAR__"]).drop_duplicates(subset=["__PROG__", "__YEAR__"])
           [["__PROG__", "__YEAR__"]].assign(has_nf=1))
dist_dim = dist.dropna(subset=["__PC4__"]).sort_values("__PC4__").drop_duplicates(subset=["__PC4__"], keep="first")

# Merge indicators (DSA & NF)
for keys, indicator in [(dsa_keys, 'has_dsa'), (nf_keys, 'has_nf')]:
    assert_no_overlap(co, keys, ignore={"__PROG__", "__YEAR__"})
    rows_before = len(co)
    co = co.merge(keys, on=["__PROG__", "__YEAR__"], how="left", validate="m:1")
    assert len(co) == rows_before
    co[indicator] = co[indicator].fillna(0).astype("Int64")

# Distance merges
dist_cols = [c for c in dist.columns if c.startswith("sdo1_distance_")]

for pc_col, prefix in [("__PC4_PREV__", "sdo1_prev_distance"), ("__PC4_HOME__", "sdo1_postal_distance")]:
    assert_no_overlap(co, dist_dim[["__PC4__"] + dist_cols], ignore={"__PC4__"})
    rows_before = len(co)
    co = co.merge(dist_dim[["__PC4__"] + dist_cols], left_on=pc_col, right_on="__PC4__", how="left", validate="m:1")
    assert len(co) == rows_before
    rename_map = {c: prefix + c[len("sdo1_distance"):] for c in dist_cols if c in co.columns}
    co.rename(columns=rename_map, inplace=True)
    co.drop(columns=["__PC4__"], inplace=True, errors="ignore")

# Cleanup
co.drop(columns=["__PROG__", "__YEAR__", "__PC4_PREV__", "__PC4_HOME__"], inplace=True, errors="ignore")

data = co
data


In [ ]:
# Verify exactly what got added and confirm uniqueness

orig_cols = set(first_merge.columns)
final_cols = set(data.columns)

added = sorted(final_cols - orig_cols)
print(f"Original columns: {len(orig_cols)}")
print(f"Final columns:    {len(final_cols)}")
print(f"Added columns:    {len(added)}")
print("\nNew columns:\n - " + "\n - ".join(added))

# Sanity: ensure no duplicates in final
assert len(data.columns) == len(set(data.columns)), "Duplicate column names found!"


##### Step 5: Verify Merge Results

Confirm that new columns were added correctly and no duplicates were created.

##### Step 6: Validate Distance Data Integrity

Verify that the two distance merges (previous school vs. home location) produced different results.

**Note:**
- `sdo1_prev_distance_*`: Distance from previous school to HU campus
- `sdo1_postal_distance_*`: Distance from student's home to HU campus

In [ ]:
# Check equality only where both exist
mask = data["sdo1_postal_distance_postcode"].notna() & data["sdo1_prev_distance_postcode"].notna()
equal_rows = (data.loc[mask, "sdo1_postal_distance_postcode"] 
              == data.loc[mask, "sdo1_prev_distance_postcode"]).sum()

total_rows = mask.sum()
print(f"Rows where both postcodes exist: {total_rows}")
print(f"Rows where they are identical:   {equal_rows}")
print(f"→ {(equal_rows / total_rows * 100):.2f}% have the same postcode")


In [ ]:
# inspect whether distances also match when postcodes match

same_dist = (
    data.loc[mask & (data["sdo1_postal_distance_postcode"] == data["sdo1_prev_distance_postcode"]),
             ["sdo1_postal_distance_distance_to_3584_CS",
              "sdo1_prev_distance_distance_to_3584_CS"]]
    .eval("diff = sdo1_postal_distance_distance_to_3584_CS - sdo1_prev_distance_distance_to_3584_CS")
    .assign(equal=lambda x: x["diff"].abs() < 1e-9)
)

print(f"Among identical postcodes: {same_dist['equal'].mean()*100:.2f}% have same distance values")


##### Step 7: Verify DSA and NF Merges

Validate that DSA and NF indicator flags were correctly assigned based on composite key matches.

In [ ]:
# Verify DSA merge correctness by comparing normalized keys
# (uses the same column spec + normalizers as the merge cell)

# --- Column names from your spec ---
co_year_col  = col_spec['co']['year']
co_prog_col  = col_spec['co']['prog']
dsa_year_col = col_spec['dsa']['year']
dsa_prog_col = col_spec['dsa']['prog']

# Recompute normalized keys for validation
co_keys = (
    data[[co_year_col, co_prog_col]]
      .assign(
          __YEAR__=norm_year(data[co_year_col]),
          __PROG__=norm_prog_code(data[co_prog_col])
      )[["__PROG__", "__YEAR__"]]
)

dsa_keys = (
    dsa[[dsa_year_col, dsa_prog_col]]
      .assign(
          __YEAR__=norm_year(dsa[dsa_year_col]),
          __PROG__=norm_prog_code(dsa[dsa_prog_col])
      )[["__PROG__", "__YEAR__"]]
)

# Apply the SAME zero-padding width rule as before (so keys compare fairly)
width = int(pd.concat([
            co_keys["__PROG__"].dropna().map(len),
            dsa_keys["__PROG__"].dropna().map(len)
        ], ignore_index=True).max() or 0)

if width > 0:
    co_keys["__PROG__"]  = co_keys["__PROG__"].str.zfill(width)
    dsa_keys["__PROG__"] = dsa_keys["__PROG__"].str.zfill(width)

print("co keys:", co_keys.dropna().drop_duplicates().shape[0])
print("dsa keys:", dsa_keys.dropna().drop_duplicates().shape[0])
print("co __PROG__ NA %:", co_keys["__PROG__"].isna().mean())
print("co __YEAR__ NA %:", co_keys["__YEAR__"].isna().mean())
print("dsa __PROG__ NA %:", dsa_keys["__PROG__"].isna().mean())
print("dsa __YEAR__ NA %:", dsa_keys["__YEAR__"].isna().mean())

# Coverage: share of co-keys in dsa-keys
coverage = (
    co_keys.dropna().drop_duplicates()
      .merge(
          dsa_keys.dropna().drop_duplicates(),
          on=["__PROG__", "__YEAR__"],
          how="left",
          indicator=True
      )["_merge"].eq("both").mean()
)

print(f"Share of co keys in sdo4_dsa: {coverage:.2%}")


##### DSA Merge Results Interpretation

- **Key Coverage:** ~20% of program-year combinations in the main dataset exist in DSA table
  - Main dataset: 323 unique (program, year) combinations
  - DSA dataset: 64 unique combinations
- **Data Quality:** No missing values in normalized keys (0% NA rate)
- **Expected Behavior:** DSA participation is optional, so low coverage is normal

In [ ]:
print(data["has_dsa"].value_counts(dropna=False))
print((data["has_dsa"]==1).mean())  # overall share of students flagged as DSA


### 4.4 Post-Merge Quality Assessment

Verify data integrity and completeness after all merge operations.

#### 4.4.1 Verify Data Completeness

Check that data from merged tables is present and correct.

In [ ]:
# Verify presence flags
flags = ["has_results", "has_orientation", "has_skc", "has_previous", "has_characteristics"]
for flag in flags:
    print(f"{flag}: {data[flag].sum(min_count=1)}")

#### 4.4.2 Missing Value Analysis

Analyze missing values across all columns in the final merged dataset.

In [ ]:
# NaN count per column in students
nan_sums = data.isna().sum().astype(int).sort_values(ascending=False)

# Convert to DataFrame for full display
nan_df = (
    nan_sums.rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)

# Show all values
nan_df.head(60)


#### 4.4.3 Data Quality Assessment Summary

**Key Findings:**
- Missing values below 12,000 per column are acceptable (less than 50% missing)
- The dataset has sufficient observations (40,000+) to handle missing values through imputation
- Data quality is adequate for proceeding to the next analysis phase

**Recommendation:** Missing values can be addressed in subsequent notebooks through appropriate imputation techniques.

In [ ]:
# Final dataset statistics
print(f"Total unique students: {data.SINH_ID.nunique()}")
print(f"Total rows: {len(data)}")
print(f"Total columns: {len(data.columns)}")

#### 4.4.4 Final Dataset Statistics

Review key statistics of the final merged dataset.

### Summary of Section 4

This section successfully completed the following data analysis and merging tasks:

**4.1 Schema Validation:**
- Created Multiple Previous Education indicator
- Validated primary keys across all datasets
- Identified datasets with valid/invalid SINH_ID keys

**4.2 Data Preparation:**
- Fixed duplicate SINH_ID values in `sdo1_previous` (kept latest exam date)
- Cleaned `sdo2_orientation` (removed invalid SINH_ID rows)
- Normalized all column names with dataset prefixes
- Re-validated schemas after cleaning
- Excluded problematic `sdo7_questionaire` dataset

**4.3 Data Merging:**
- **First Merge:** Combined 6 datasets using SINH_ID (47,950 students)
  - Base: sdo5_degree + characteristics + previous + SKC + orientation + results
  - Added presence flags for each merged dataset
- **Second Merge:** Added 3 datasets using composite keys
  - DSA participation (program + year)
  - NF data (program + year)  
  - Distance data (postcode - merged twice for home & previous school)
  
**4.4 Quality Assessment:**
- Verified merge integrity (row counts preserved)
- Validated distance data differentiation
- Confirmed DSA/NF indicator correctness (~20% DSA coverage)
- Assessed missing values (all below 50% threshold)
- Final dataset: 40,000+ students with comprehensive data

The merged dataset is now ready for export and subsequent cleaning/feature engineering steps.

---

## 5. Final Dataset Export {#final-dataset-export}

Save the combined and cleaned dataset for use in subsequent analysis notebooks.

In [ ]:
# save combined data frame
out_path = OUT_DIR / "v2_combined.csv"
data.to_csv(out_path, index=False)